# Models

And now - this colab unveils the heart (or the brains?) of the transformers library - the models:

https://colab.research.google.com/drive/1hhR9Z-yiqjUe7pJjVQw4c74z_V3VchLy?usp=sharing

This should run nicely on a low-cost or free T4 box.

#### **The most important function:**

In [ ]:
# Wrapping everything in a function - and adding Streaming and generation prompts

def generate(model, messages, quant=True, max_new_tokens=80):
  tokenizer = AutoTokenizer.from_pretrained(model)
  tokenizer.pad_token = tokenizer.eos_token
  input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")

  # The general job of attention_mask is to tell the model which token positions are real content versus padding. 
  # When you batch multiple prompts of different lengths together, the tokenizer pads the shorter ones out to match the longest, 
  # and the mask marks those padding positions with 0 so two things happen correctly: the attention computation excludes padded 
  # positions entirely (rather than letting real tokens attend to meaningless pad tokens), and generate() computes position_ids 
  # based on the actual content length rather than the padded length, which matters for getting the right rotary position embeddings.
  attention_mask = torch.ones_like(input_ids, dtype=torch.long, device="cuda")
  streamer = TextStreamer(tokenizer)
  if quant:
    model = AutoModelForCausalLM.from_pretrained(model, quantization_config=quant_config).to("cuda")
  else:
    model = AutoModelForCausalLM.from_pretrained(model).to("cuda")
  outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, temperature = 0.7, streamer=streamer)